In [1]:
!pip install -q "qiskit>=1.0.0" "qiskit-aer>=0.14.0" matplotlib pylatexenc

import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, partial_trace
from qiskit_aer import AerSimulator

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 2.9 MB/s eta 0:00:00


In [2]:
# Construct |Phi+>
qc_phi_plus = QuantumCircuit(2, 2)
qc_phi_plus.h(0)
qc_phi_plus.cx(0, 1)
qc_phi_plus.measure([0, 1], [0, 1])

# Simulate
simulator = AerSimulator()
counts_phi_plus = simulator.run(qc_phi_plus, shots=1024).result().get_counts()

print("Exercise 1 (|Phi+>) Measurement Counts (1024 shots):", counts_phi_plus)
print("Confirmed: Only '00' and '11' observed?", set(counts_phi_plus.keys()).issubset({'00', '11'}))
print("\nCircuit:")
print(qc_phi_plus.draw('text'))

Exercise 1 (|Phi+>) Measurement Counts (1024 shots): {'11': 505, '00': 519}
Confirmed: Only '00' and '11' observed? True

Circuit:
     ┌───┐     ┌─┐   
q_0: ┤ H ├──■──┤M├───
     └───┘┌─┴─┐└╥┘┌─┐
q_1: ─────┤ X ├─╫─┤M├
          └───┘ ║ └╥┘
c: 2/═══════════╩══╩═
                0  1 


In [3]:
def create_bell_circuit(state_name):
    qc = QuantumCircuit(2, 2)

    # State preparation
    if state_name == "Phi+":
        pass
    elif state_name == "Phi-":
        qc.x(0)
    elif state_name == "Psi+":
        qc.x(1)
    elif state_name == "Psi-":
        qc.x(0)
        qc.x(1)
    else:
        raise ValueError("Invalid Bell state name")

    # Bell basis transformation
    qc.h(0)
    qc.cx(0, 1)
    qc.measure([0, 1], [0, 1])
    return qc

bell_states = ["Phi+", "Phi-", "Psi+", "Psi-"]
simulator = AerSimulator()

print(f"{'Bell State':<12} | {'Counts (1024 shots)':<30} | {'Expected Basis States'}")
print("-" * 65)

for b_state in bell_states:
    qc = create_bell_circuit(b_state)
    counts = simulator.run(qc, shots=1024).result().get_counts()
    expected = "{'00', '11'}" if "+" in b_state and "Phi" in b_state or "-" in b_state and "Phi" in b_state else "{'01', '10'}"
    print(f"{b_state:<12} | {str(counts):<30} | {expected}")

Bell State   | Counts (1024 shots)            | Expected Basis States
-----------------------------------------------------------------
Phi+         | {'11': 493, '00': 531}         | {'00', '11'}
Phi-         | {'11': 510, '00': 514}         | {'00', '11'}
Psi+         | {'10': 509, '01': 515}         | {'01', '10'}
Psi-         | {'10': 493, '01': 531}         | {'01', '10'}


In [4]:
def measure_in_hadamard_basis(state_name="Phi+"):
    qc = QuantumCircuit(2, 2)

    # Prepare Bell state
    if state_name == "Phi-":
        qc.x(0)
    qc.h(0)
    qc.cx(0, 1)
    qc.barrier()

    # Change of measurement basis: Z -> X via H gates
    qc.h(0)
    qc.h(1)
    qc.barrier()

    qc.measure([0, 1], [0, 1])
    return qc

simulator = AerSimulator()

for name in ["Phi+", "Phi-"]:
    qc_x = measure_in_hadamard_basis(name)
    counts = simulator.run(qc_x, shots=1024).result().get_counts()
    print(f"|{name}> measured in X-basis counts:", counts)
    print(f"Explanation: |{name}> in X-basis manifests as correlation ({'00/11' if name=='Phi+' else '01/10'})\n")

print("Circuit Diagram for X-basis measurement:")
print(qc_x.draw('text'))

|Phi+> measured in X-basis counts: {'11': 534, '00': 490}
Explanation: |Phi+> in X-basis manifests as correlation (00/11)

|Phi-> measured in X-basis counts: {'01': 533, '10': 491}
Explanation: |Phi-> in X-basis manifests as correlation (01/10)

Circuit Diagram for X-basis measurement:
     ┌───┐┌───┐      ░ ┌───┐ ░ ┌─┐   
q_0: ┤ X ├┤ H ├──■───░─┤ H ├─░─┤M├───
     └───┘└───┘┌─┴─┐ ░ ├───┤ ░ └╥┘┌─┐
q_1: ──────────┤ X ├─░─┤ H ├─░──╫─┤M├
               └───┘ ░ └───┘ ░  ║ └╥┘
c: 2/═══════════════════════════╩══╩═
                                0  1 


In [5]:
def simulate_e91_channel(eavesdropper_present=False):
    """
    Simulates a shared |Phi+> pair between Alice and Bob.
    If Eve eavesdrops, she performs an uncoordinated projective measurement,
    collapsing the entanglement and introducing errors into Alice & Bob's correlations.
    """
    qc = QuantumCircuit(2, 2)
    # Entangled source sends qubit 0 to Alice, qubit 1 to Bob
    qc.h(0)
    qc.cx(0, 1)

    if eavesdropper_present:
        # Eve intercepts Bob's qubit and measures it in Z-basis
        qc.barrier()
        qc.measure(1, 1) # Projective measurement collapses the state
        qc.barrier()

    # Alice and Bob both choose the Hadamard (X) basis to test correlation
    qc.h(0)
    qc.h(1)
    qc.measure([0, 1], [0, 1])

    simulator = AerSimulator()
    counts = simulator.run(qc, shots=1024).result().get_counts()
    return counts

counts_secure = simulate_e91_channel(eavesdropper_present=False)
counts_eavesdropped = simulate_e91_channel(eavesdropper_present=True)

# In X-basis, intact |Phi+> only yields '00' or '11'.
# Eve's measurement produces '01' and '10' errors (QBER ~ 25-50%).
def compute_qber(counts):
    total = sum(counts.values())
    errors = counts.get('01', 0) + counts.get('10', 0)
    return (errors / total) * 100

print("--- QKD Channel Security Demonstration (Ekert91 Basis) ---")
print("Uncompromised Channel Counts (X-basis):", counts_secure)
print(f"Quantum Bit Error Rate (QBER): {compute_qber(counts_secure):.2f}% (Expected: 0%)\n")

print("Compromised Channel (Eve eavesdropping) Counts:", counts_eavesdropped)
print(f"Quantum Bit Error Rate (QBER): {compute_qber(counts_eavesdropped):.2f}% (Detectable anomaly!)")

--- QKD Channel Security Demonstration (Ekert91 Basis) ---
Uncompromised Channel Counts (X-basis): {'00': 519, '11': 505}
Quantum Bit Error Rate (QBER): 0.00% (Expected: 0%)

Compromised Channel (Eve eavesdropping) Counts: {'11': 208, '01': 273, '10': 278, '00': 265}
Quantum Bit Error Rate (QBER): 53.81% (Detectable anomaly!)


In [6]:
# 1. Prepare GHZ State
ghz_circuit = QuantumCircuit(3)
ghz_circuit.h(0)
ghz_circuit.cx(0, 1)
ghz_circuit.cx(1, 2)
ghz_sv = Statevector.from_instruction(ghz_circuit)

# 2. Prepare W-State
w_circuit = QuantumCircuit(3)
# Exact gate decomposition for equal superposition 1/sqrt(3)
theta = 2 * np.arccos(1 / np.sqrt(3))
w_circuit.ry(theta, 0)
w_circuit.ch(0, 1)
w_circuit.cx(1, 2)
w_circuit.cx(0, 1)
w_circuit.x(0)
w_sv = Statevector.from_instruction(w_circuit)

# 3. Analyze partial trace (tracing out qubit 2)
# Tracing out qubit 2 gives the reduced density matrix rho_01 of the remaining pair
rho_ghz_subsystem = partial_trace(ghz_sv, [2])
rho_w_subsystem = partial_trace(w_sv, [2])

def purity(rho):
    """Calculates Tr(rho^2). Purity = 1 indicates pure state, < 1 indicates mixed state."""
    return np.real(np.trace(rho.data @ rho.data))

print("=== Entanglement Structure & Subsystem Analysis ===")
print("GHZ Statevector:", np.round(ghz_sv.data, 3))
print("W-State Statevector:", np.round(w_sv.data, 3))
print("-" * 55)

print(f"GHZ Reduced Density Matrix (Qubits 0 & 1):\n{np.round(rho_ghz_subsystem.data, 3)}")
print(f"Purity: {purity(rho_ghz_subsystem):.2f}")
print("-> GHZ verdict: Subsystem is a completely classical statistical mixture: 0.5|00><00| + 0.5|11><11|.")
print("   All bipartite entanglement between qubits 0 and 1 is destroyed.\n")

print(f"W-State Reduced Density Matrix (Qubits 0 & 1):\n{np.round(rho_w_subsystem.data, 3)}")
print(f"Purity: {purity(rho_w_subsystem):.2f}")
print("-> W-state verdict: Off-diagonal coherence terms (rho_01,10 != 0) remain non-zero.")
print("   Subsystem retains residual bipartite entanglement even after losing a qubit.")

=== Entanglement Structure & Subsystem Analysis ===
GHZ Statevector: [0.707+0.j 0.   +0.j 0.   +0.j 0.   +0.j 0.   +0.j 0.   +0.j 0.   +0.j
 0.707+0.j]
W-State Statevector: [0.   +0.j 0.577+0.j 0.577+0.j 0.   +0.j 0.577+0.j 0.   +0.j 0.   +0.j
 0.   +0.j]
-------------------------------------------------------
GHZ Reduced Density Matrix (Qubits 0 & 1):
[[0.5+0.j 0. +0.j 0. +0.j 0. +0.j]
 [0. +0.j 0. +0.j 0. +0.j 0. +0.j]
 [0. +0.j 0. +0.j 0. +0.j 0. +0.j]
 [0. +0.j 0. +0.j 0. +0.j 0.5+0.j]]
Purity: 0.50
-> GHZ verdict: Subsystem is a completely classical statistical mixture: 0.5|00><00| + 0.5|11><11|.
   All bipartite entanglement between qubits 0 and 1 is destroyed.

W-State Reduced Density Matrix (Qubits 0 & 1):
[[0.333+0.j 0.   +0.j 0.   +0.j 0.   +0.j]
 [0.   +0.j 0.333+0.j 0.333+0.j 0.   +0.j]
 [0.   +0.j 0.333+0.j 0.333+0.j 0.   +0.j]
 [0.   +0.j 0.   +0.j 0.   +0.j 0.   +0.j]]
Purity: 0.56
-> W-state verdict: Off-diagonal coherence terms (rho_01,10 != 0) remain non-zero.
   Subs